# 0. Bibliotecas

In [1]:
import os
from dotenv import load_dotenv
import chromadb
import chromadb.api
import logging

# Importações do ecossistema LangChain
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
 

# 1. Configurações iniciais

In [ ]:
# Carrega a OPENAI_API_KEY e demais variáveis definidas no arquivo .env
load_dotenv()

# Suprime os logs de erro do módulo de telemetria do ChromaDB
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)

# Caminhos dos PDFs que serão indexados
PDF_PATHS = ["files/apostila.pdf", "files/LLM.pdf"]

# Diretório onde o banco vetorial será salvo/carregado
VECTOR_DB_DIR = "arquivos/chat_retrieval_db"

# 2. Ingestão e chunking dos documentos

In [9]:
def ingest_documents(paths: list[str]):
    """
    Carrega PDFs e os divide em chunks menores (pedaços de texto).
    
    O chunking é necessário porque LLMs têm uma janela de contexto limitada.
    Em vez de enviar o documento inteiro, enviamos apenas os trechos mais
    relevantes para cada pergunta — isso reduz custo e melhora a precisão.
    
    Parâmetros de chunking:
    - chunk_size: tamanho máximo de cada trecho (em caracteres)
    - chunk_overlap: sobreposição entre trechos consecutivos, para não
      perder contexto nas bordas de um chunk
    """
    all_pages = []
    for path in paths:
        loader = PyPDFLoader(path)
        all_pages.extend(loader.load())

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    docs = text_splitter.split_documents(all_pages)

    # Padroniza o nome do arquivo nos metadados e adiciona um ID único por chunk
    # Esses metadados serão exibidos na etapa de citação de fontes
    for i, doc in enumerate(docs):
        doc.metadata["source"] = doc.metadata["source"].replace("files/", "")
        doc.metadata["doc_id"] = i

    return docs

documents = ingest_documents(PDF_PATHS)
print(f"{len(documents)} chunks criados a partir dos PDFs.")

Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 149 0 (offset 0)
Ignoring wrong pointing object 155 0 (offset 0)
Ignoring wrong pointing object 158 0 (offset 0)
Ignoring wrong pointing object 160 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignori

81 chunks criados a partir dos PDFs.


# 3. Embeddings e banco vetorial

In [10]:
print("Carregando modelo de embeddings...")

# Modelo local de embeddings (roda na CPU, sem custo de API)
# Converte cada chunk de texto em um vetor numérico para busca por similaridade
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Em notebooks, re-executar a célula causaria ValueError pois o ChromaDB
# mantém instâncias em memória no kernel. O clear_system_cache() evita isso.
chromadb.api.client.SharedSystemClient.clear_system_cache()
persistent_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)

# Verifica se já existe um banco salvo para evitar reprocessar os PDFs
if os.path.exists(VECTOR_DB_DIR) and len(persistent_client.list_collections()) > 0:
    print("Banco vetorial encontrado — carregando...")
    vectordb = Chroma(
        client=persistent_client,
        embedding_function=embeddings_model
    )
else:
    print("Nenhum banco encontrado — criando do zero...")
    documents = ingest_documents(PDF_PATHS)
    vectordb = Chroma.from_documents(
        documents=documents,
        embedding=embeddings_model,
        client=persistent_client
    )
    print("Banco vetorial criado e salvo com sucesso.")

Carregando modelo de embeddings...
Banco vetorial encontrado — carregando...


# 4. LLM e prompt

In [11]:
# GPT-4o-mini: ótimo custo-benefício para tarefas de QA sobre documentos
# temperature=0 garante respostas determinísticas e factuais (sem "criatividade")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# O prompt instrui o modelo a responder apenas com base no contexto recuperado,
# evitando alucinações. O {context} é preenchido automaticamente pela chain
# com os chunks mais relevantes; o {question} recebe a pergunta do usuário.
template = """Utilize o contexto fornecido para responder a pergunta ao final.
Se você não sabe a resposta, apenas diga que não sabe e não invente uma resposta.
Utilize três frases no máximo e mantenha a resposta concisa.

Contexto: {context}

Pergunta: {question}

Resposta:"""

custom_prompt = PromptTemplate.from_template(template)

# 5. Criação da chain RAG

In [12]:
# RetrievalQA orquestra o fluxo RAG completo:
#   1. Recebe a pergunta
#   2. Busca os chunks mais relevantes no banco vetorial (retriever)
#   3. Injeta esses chunks no prompt como {context}
#   4. Envia o prompt preenchido ao LLM
#   5. Retorna a resposta e os documentos fonte

# search_type="mmr": Maximum Marginal Relevance — equilibra relevância e
# diversidade nos chunks retornados, evitando trechos muito repetitivos
# k=3: número de chunks recuperados por consulta

chat_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3}
    ),
    chain_type="stuff",  # "stuff": concatena todos os chunks em um único prompt
    chain_type_kwargs={"prompt": custom_prompt},
    return_source_documents=True  # permite auditar quais trechos embasaram a resposta
)

# 6. Execução e testes

In [13]:
pergunta = "O que é Hugging Face e como faço para acessá-lo?"
print(f"Pergunta: {pergunta}\n")

resposta = chat_chain.invoke({"query": pergunta})

print("--- RESPOSTA ---")
print(resposta["result"])

print("\n--- FONTES CONSULTADAS ---")
for doc in resposta["source_documents"]:
    print(f"  • {doc.metadata['source']} (chunk ID: {doc.metadata['doc_id']})")
    
    


Pergunta: O que é Hugging Face e como faço para acessá-lo?

--- RESPOSTA ---
Hugging Face é uma comunidade de código aberto que reúne modelos de linguagem para diversas aplicações, como geração de texto e classificação. Você pode acessá-lo através do site oficial da Hugging Face, onde encontrará modelos e ferramentas disponíveis para uso. É uma ótima opção para quem busca soluções de linguagem natural sem custos elevados.

--- FONTES CONSULTADAS ---
  • files/LLM.pdf (chunk ID: 73)
  • files/apostila.pdf (chunk ID: 46)
  • files/apostila.pdf (chunk ID: 10)


In [14]:
pergunta = "Quais são as diferenças entre GPT e outros modelos de linguagem?"
print(f"Pergunta: {pergunta}\n")

resposta = chat_chain.invoke({"query": pergunta})

print("--- RESPOSTA ---")
print(resposta["result"])

print("\n--- FONTES CONSULTADAS ---")
for doc in resposta["source_documents"]:
    print(f"  • {doc.metadata['source']} (chunk ID: {doc.metadata['doc_id']})")
    
    


Pergunta: Quais são as diferenças entre GPT e outros modelos de linguagem?

--- RESPOSTA ---
Os modelos GPT, como o GPT-3 e GPT-4, são conhecidos por seu tamanho e desempenho superiores, com bilhões de parâmetros, permitindo tarefas complexas de linguagem. Em contraste, outros modelos podem ter arquiteturas diferentes ou menos parâmetros, resultando em desempenho inferior. Além disso, os modelos GPT geralmente requerem mais recursos computacionais e levantam preocupações sobre privacidade e controle dos dados dos usuários.

--- FONTES CONSULTADAS ---
  • files/LLM.pdf (chunk ID: 74)
  • files/LLM.pdf (chunk ID: 61)
  • files/LLM.pdf (chunk ID: 72)
